<a href="https://colab.research.google.com/github/FabioContrera/criando-agentes-de-ia-com-agno/blob/main/Aula%204/Aula_4_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 4.1 — O que são workflows e quando usar

**Curso:** Agno: Criando agentes e sistemas multiagente

**Aula:** 4 — Orquestrando agentes com workflows

**Instrutor:** Fabio Contrera

---

## Onde estamos

Na **Aula 3** construímos um time conversacional do ScoutAI FC:
- Treinador-líder
- Olheiro
- Analista

Nova demanda:

**Ex.: Recomendação de escalação**:
```
1. Coletar dados dos jogadores
2. Analisar forma
3. Decidir escalação
4. Apresentar
```

Plano da aula:

1. **Conceito de workflow** em poucas linhas
2. Implementar o **workflow mínimo**


## Setup


In [1]:
!pip install -U agno openai tavily-python wikipedia

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 100.6 MB/s eta 0:00:00
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=dba73e9652032c9bf85a83644820dbcfd80af4ca6a4d8d5cb794fdd9cb0e0328
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia
  Attempting uninstall: plotly
    Found existing installation: plotly 5.24.1
    Uninstalling plotly-5.24.1:
      Successfully uninstalled plotly-5.24.1
  Attempting uninstall: openai
    Found existing installation: openai 2.32.0
    Uninstalling openai-2.32.0:
      Successfully uninstalled openai-2.32.0


In [2]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

---

## Passo 1 — Reconstruindo os agentes da Aula 3



In [3]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.team import Team
from agno.tools.tavily import TavilyTools
from agno.tools.wikipedia import WikipediaTools

# Modelos por agente (boa prática estabelecida na Aula 3.3)
modelo_olheiro = OpenAIChat(id="gpt-5.4")
modelo_treinador = OpenAIChat(id="gpt-5.4")

olheiro = Agent(
    name="Olheiro",
    role="Busca informação verificável sobre futebol em fontes externas e em base interna oficial",
    model=modelo_olheiro,
    instructions=[
        "Você é o Olheiro do ScoutAI FC. Sua função é buscar informação verificável.",
        "Você tem duas fontes disponíveis:",
        #"• BASE INTERNA (Knowledge): regulamento oficial da Copa do Mundo 2026 — formato, critérios de desempate, regras de fase eliminatória.",
        "• Tavily (web): eventos recentes — últimos jogos, convocações, forma de jogadores.",
        "• Wikipedia: fatos históricos consolidados — Copas antigas, biografias, técnicos do passado.",
        #"Sempre que a pergunta envolver REGRAS OFICIAIS da Copa 2026, busque PRIMEIRO na base interna — é a fonte autoritativa.",
        "Para forma atual, use Tavily. Para histórico, use Wikipedia. Combine fontes quando a pergunta tiver múltiplas necessidades.",
        "Retorne dados objetivos — não interprete nem opine.",
        "Se a busca falhar em todas as fontes, diga claramente em vez de chutar.",
    ],
    tools=[TavilyTools(), WikipediaTools()],
    #knowledge=knowledge,            # ← peça nova: base interna como knowledge
    #search_knowledge=True,          # ← permite ao agente buscar na base
    markdown=True,
)

# Treinador agora é um Agent comum, não líder de time.
# Ele recebe os dados da etapa anterior e produz a resposta final ao usuário.
treinador = Agent(
    name="Treinador do ScoutAI FC ",
    role="Sintetiza dados e produz recomendação tática para o usuário",
    model=modelo_treinador,
    instructions=[
        "Você é o Treinador do ScoutAI FC — uma plataforma de IA dedicada à Seleção Brasileira masculina de futebol.",
        "Sua especialidade é a Seleção: história, conquistas, jogadores, técnicos, táticas e cultura ao redor do time.",
        "Seu público são torcedores, analistas e comissões técnicas. Adapte o nível técnico ao tipo de pergunta.",

        "Sua função: produzir uma recomendação de escalação clara e estruturada.",
        "Sempre inclua: formação sugerida (ex: 4-3-3), 11 titulares com posição, "
        "principais reservas e justificativa tática em 2-3 frases.",
        "Responda em português do Brasil, com tom profissional.",
    ],
    markdown=True,
)

---

## Passo 2 — O que é um workflow?

```
Pergunta do usuário
        │
        ▼
   ETAPA 1: Olheiro coleta dados
        │ (output → input)
        ▼
   ETAPA 2: Treinador sintetiza e apresenta
        │
        ▼
   Resposta
```

A diferença em relação ao time:

| Aspecto | Time conversacional (Aula 3) | Workflow (esta aula) |
|---|---|---|
| **Quem decide o fluxo** | LLM (líder do time) | Código |
| **Ordem das etapas** | Variável a cada execução | Fixa, sempre a mesma |
| **Previsibilidade** | Baixa-média | Alta |
| **Flexibilidade** | Alta | Baixa-média |
| **Auditabilidade** | Difícil | Direta |
| **Quando usar** | Conversa, perguntas exploratórias | Tarefas estruturadas, decisões repetidas |



---

## Passo 3 — Construindo o workflow mínimo

O Agno tem duas peças centrais pra workflow:

- **`Step`** — uma etapa. Aceita `agent=...` (agente que executa), `team=...` (time que executa) ou `executor=função` (função Python custom)
- **`Workflow`** — o orquestrador. Recebe lista de `Step` em `steps=[...]`, executa em ordem.



In [4]:
from agno.workflow import Step, Workflow

workflow_escalacao = Workflow(
    name="Recomendação de Escalação",
    steps=[
        Step(name="Coleta de dados", agent=olheiro),
        Step(name="Síntese da recomendação", agent=treinador),
    ],
)

---

## Passo 4 — O workflow em ação



In [5]:
workflow_escalacao.print_response(
"Recomende a escalação ideal da Seleção pro próximo jogo (Brasil vs Panamá), "
"considerando forma física atual dos convocados e o adversário. Exclua da lista jogadores lesionados.",
    stream=True,
)

┏━ Workflow Information ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ Workflow: Recomendação de Escalação                                                                             ┃
┃                                                                                                                 ┃
┃ Steps: 2 steps                                                                                                  ┃
┃                                                                                                                 ┃
┃ Message: Recomende a escalação ideal da Seleção pro próximo jogo (Brasil vs Panamá), considerando forma física  ┃
┃ atual dos convocados e o adversário. Exclua da lista jogadores lesionados.                                      ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

Output()

┏━ Step 1: Coleta de dados (Completed) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ Não consigo atender exatamente ao pedido de “recomendar a escalação ideal” porque isso exige                    ┃
┃ interpretação/opinião, e minha função aqui é trazer informação verificável.                                     ┃
┃                                                                                                                 ┃
┃ Mas posso montar uma base objetiva para a escalação do Brasil contra o Panamá, excluindo lesionados e           ┃
┃ destacando o que as fontes indicam sobre condição física e perfil do adversário.                                ┃
┃                                                                                                                 ┃
┃                                                                                                                 ┃
┃                            O que as fontes indicam### Próximo jogo- Brasil x Panamá                             ┃
┃                                                                                                                 ┃
┃  • Data:31 de maio de2026- Local: MaracanãFonte: ge / Brazilian Times via Tavily### Critério físico usado por   ┃
┃    AncelottiNa convocação de março de2026, Carlo Ancelotti afirmou que a lista foi feita com jogadores em “100% ┃
┃    de condição física”.                                                                                         ┃
┃                                                                                                                 ┃
┃        Lesionados citados nas fontesNa cobertura da convocação, foram mencionados como baixas por lesão:        ┃
┃                                                                                                                 ┃
┃  • Éder Militão                                                                                                 ┃
┃  • Bruno Guimarães                                                                                              ┃
┃  • Estêvão                                                                                                      ┃
┃  • Rodrygo                                                                                                      ┃
┃                                                                                                                 ┃
┃ Além disso:                                                                                                     ┃
┃                                                                                                                 ┃
┃  • Neymar ficou fora por não estar em condição física ideal, segundo as reportagens consultadas.Fonte: ge via   ┃
┃    Tavily---                                                                                                    ┃
┃                                                                                                                 ┃
┃                                                                                                                 ┃
┃          Convocados citados nas fontes de março/2026As fontes consultadas mencionam entre os chamados:          ┃
┃                                                                                                                 ┃
┃  • Vini Jr.                                                                                                     ┃
┃  • Endrick                                                                                                      ┃
┃  • Rayan                                                                                                        ┃
┃  • Danilo                                                                                                       ┃
┃  • Ibañez                                             

┏━ Step 2: Síntese da recomendação (Completed) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃                                                                                                                 ┃
┃                                           Formação sugerida: 4-2-3-1                                            ┃
┃                                                                                                                 ┃
┃                                                  11 titulares                                                   ┃
┃                                                                                                                 ┃
┃  • Goleiro: Alisson- Lateral-direito: Danilo- Zagueiro: Ibañez- Zagueiro: Léo Pereira- Lateral-esquerdo:        ┃
┃    Guilherme Arana- Volante: João Gomes- Volante: Gabriel Sara- Meia central: Lucas Paquetá- Ponta-direita:     ┃
┃    Rayan- Ponta-esquerda: Vini Jr.- Centroavante: Endrick### Principais reservas                                ┃
┃  • Bento (goleiro)- Vanderson (lateral-direito)- Bremer (zagueiro)- Carlos Augusto (lateral-esquerdo)- André    ┃
┃    (volante)- Andreas Pereira (meia)- Savinho (ponta)- Igor Thiago (centroavante)### Justificativa tática       ┃
┃    Contra um Panamá que tende a defender em bloco baixo e explorar transições, o ideal é um Brasil com          ┃
┃    amplitude forte pelos lados, circulação rápida por dentro e pressão pós-perda. Vini Jr. e Rayan dão          ┃
┃    profundidade, enquanto Paquetá atua entrelinhas para acionar Endrick em zona de finalização. A dupla João    ┃
┃    Gomes + Gabriel Sara oferece equilíbrio para sustentar o ataque e proteger o time contra contra-ataques.     ┃
┃                                                                                                                 ┃
┃                                              Observação importante                                              ┃
┃                                                                                                                 ┃
┃ Como você mesmo apontou, a lista final oficial para o jogo e a condição física atualizada de todos os atletas   ┃
┃ podem alterar essa projeção. Então, esta é uma recomendação tática provável, não uma confirmação oficial.Se     ┃
┃ quiser, eu posso montar também uma versão mais conservadora e uma versão mais ofensiva dessa escalação para     ┃
┃ enfrentar o Panamá.                                                                                             ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

Completed in 38.8s


### Estado atual do produto

```
ScoutAI FC (estado atual)
│
├── ✅ Time conversacional (Aula 3) — para perguntas exploratórias
├── ✅ Workflow mínimo (esta aula) — para tarefas estruturadas
│   ├── Step 1: Coleta (Olheiro)
│   └── Step 2: Síntese (Treinador)
│
├── ❌ Sem agente especialista em decidir escalação      → Aula 4.2 (Auxiliar Técnico)
├── ❌ Sem contrato de tipos entre etapas (Pydantic)     → Aula 4.2
├── ❌ Sem lógica condicional, paralela ou iterativa     → Aula 4.3
└── ❌ Sem roteamento entre time e workflow              → Aula 4.3
```

### Próxima aula

**Aula 4.2 — Construindo o workflow completo**

